<a href="https://colab.research.google.com/github/tpedCode/P05/blob/main/BARRET_Marjorie_3_script_API_202606.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Identifiant externalisé via une variable d’environnement
headers = {
    "User-Agent": os.getenv("USER_AGENT")
}

In [ ]:
import requests
import pandas as pd
import time

url = "https://world.openfoodfacts.org/cgi/search.pl"

# =========================
# 1. Appel API avec fallback
# =========================

page_sizes = [25, 15, 10]

data = None

for size in page_sizes:
    print(f"Tentative avec page_size={size}")

    params = {
        "search_terms": "champagne",
        "search_simple": 1,
        "action": "process",
        "json": 1,
        "page_size": size
    }

    for i in range(3):
        try:
            response = requests.get(url, params=params, headers=headers)

            if response.status_code == 200:
                print(f"Tentative {i+1} : 200 OK → requête réussie")
                data = response.json()
                break

            elif response.status_code == 503:
                print(f"Tentative {i+1} : 503 Service Unavailable → serveur indisponible, retry...")

            elif response.status_code == 403:
                print(f"Tentative {i+1} : 403 Forbidden → accès refusé, retry...")

            else:
                print(f"Tentative {i+1} : {response.status_code} → erreur inattendue, retry...")

            time.sleep(2)

        except requests.exceptions.RequestException as e:
            print(f"Tentative {i+1} : erreur réseau → {e}")
            time.sleep(2)

    if data is not None:
        break

# Vérification finale
if data is None:
    raise Exception("Impossible de récupérer les données")

products = data.get("products", [])
print("Produits récupérés (avant filtrage) :", len(products))


# =========================
# 2. Filtrage
# =========================

allowed_keywords = ["champagne", "vin", "boisson"]

filtered_products = []

for p in products:
    name = (p.get("product_name") or "").lower()
    category = (p.get("categories") or "").lower()
    image = p.get("image_url")

    if not name or not category or not image:
        continue

    if not any(k in category for k in allowed_keywords):
        continue

    if "miel" in name:
        continue

    filtered_products.append(p)

print("Nombre de produits après filtrage :", len(filtered_products))

# Sélection des 10 premiers
final_products = filtered_products[:10]
print("Nombre de produits sélectionnés :", len(final_products))


# =========================
# 3. Transformation
# =========================

results = []

for p in final_products:
    results.append({
        "foodId": p.get("code"),
        "label": p.get("product_name"),
        "category": p.get("categories"),
        "foodContentsLabel": p.get("ingredients_text"),
        "image": p.get("image_url")
    })

df = pd.DataFrame(results)

print("Nombre de lignes dans le DataFrame :", len(df))


# =========================
# 4. Sauvegarde
# =========================

output_path = "produits_champagne.csv"
df.to_csv(output_path, index=False)

print("Fichier créé :", output_path)

Tentative avec page_size=25
Tentative 1 : 503 Service Unavailable → serveur indisponible, retry...
Tentative 2 : 503 Service Unavailable → serveur indisponible, retry...
Tentative 3 : 200 OK → requête réussie
Produits récupérés (avant filtrage) : 25
Nombre de produits après filtrage : 11
Nombre de produits sélectionnés : 10
Nombre de lignes dans le DataFrame : 10
Fichier créé : produits_champagne.csv
